<a href="https://colab.research.google.com/github/oselumeseagbonrofo/small-llm-experiments/blob/main/full_finetuning_gpt2_small.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Full Finetuning with GPT2-small to write manim code

## Load Dataset

In [1]:
from datasets import load_dataset
dataset = load_dataset("Edoh/manim_python")

README.md:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/135k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/11.3k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/599 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/51 [00:00<?, ? examples/s]

## Load Model tokenizer

In [2]:
from transformers import GPT2Tokenizer
model_name = "openai-community/gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

## Implement preprocessing function

In [3]:
def preprocess_data(examples):
 inputs = [
 f"Instruction: {instr}\nOutput: {out}"
 for instr, out in zip(examples["instruction"], examples["output"])
 ]

 tokenized = tokenizer(inputs, truncation=True, max_length=512,
                       padding="max_length")

 tokenized["labels"] = tokenized["input_ids"].copy()
 return tokenized

tokenized_datasets = dataset.map(preprocess_data,
 batched=True,
 remove_columns=dataset["train"].column_names)

Map:   0%|          | 0/599 [00:00<?, ? examples/s]

Map:   0%|          | 0/51 [00:00<?, ? examples/s]

## Initialise model

In [4]:
from transformers import GPT2LMHeadModel
def model_init():
 return GPT2LMHeadModel.from_pretrained(model_name, device_map='auto')

## Configure training arguments for hyperparameter search

In [5]:
from transformers import TrainingArguments
training_args = TrainingArguments(
 output_dir="./gpt2-manim-python-finetuned",
 eval_strategy="epoch",
 save_strategy="epoch",
 logging_strategy="steps",
 logging_steps=100,
 save_total_limit=2,
 load_best_model_at_end=True,
 metric_for_best_model="eval_loss",
 greater_is_better=False,
 fp16=True,
 report_to="none",
)

In [6]:
train_val_split = tokenized_datasets["train"].train_test_split(test_size=0.1)
tokenized_datasets["train"] = train_val_split["train"]
tokenized_datasets["validation"] = train_val_split["test"]

## Initialise trainer

In [8]:
from transformers import (
 Trainer,
 DataCollatorForLanguageModeling,
 EarlyStoppingCallback
)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer,
 mlm=False)
trainer = Trainer(
 model_init=model_init,
 args=training_args,
 train_dataset=tokenized_datasets["train"],
 eval_dataset=tokenized_datasets["validation"],
 data_collator=data_collator,
 callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## Define hyperparameter tuning search space

In [9]:
def hp_space(trial):
 return {
 "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True),
 "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size",
  [2, 4, 8]),
 "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.3),
 "num_train_epochs": trial.suggest_int("num_train_epochs", 3, 6),
 "warmup_steps": trial.suggest_int("warmup_steps", 0, 500),
 "gradient_accumulation_steps": trial.suggest_categorical("gradient_accumulation_steps",
  [1, 2, 4]),
 }

## Run hyperparameter search

In [11]:
!pip install -q optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 23.9 MB/s eta 0:00:00


In [12]:
best_run = trainer.hyperparameter_search(
 direction="minimize",
 backend="optuna",
 n_trials=10,
 hp_space=hp_space,
 compute_objective=lambda metrics: metrics["eval_loss"],
)

[I 2026-08-16 22:06:20,273] A new study created in memory with name: no-name-b12562f2-bb98-4be4-8b46-fa5244ef736f


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,2.794424,0.603264
2,0.733199,0.232214
3,0.261585,0.172523
4,0.222398,0.149053
5,0.193441,0.134911
6,0.158056,0.130033


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[I 2026-08-16 22:35:02,576] Trial 8 pruned. 


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,No log,1.274137


[I 2026-08-16 22:35:34,313] Trial 9 pruned. 


## Configure trainer with best hyperparameters

In [14]:
for key, value in best_run.hyperparameters.items():
 setattr(training_args, key, value)
trainer = Trainer(
 model_init=model_init,
 args=training_args,
 train_dataset=tokenized_datasets["train"],
 eval_dataset=tokenized_datasets.get("validation"),
 data_collator=data_collator,
 callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [15]:
trainer.train()
trainer.save_model("./gpt2-manim-python-finetuned")
tokenizer.save_pretrained("./gpt2-manim-python-finetuned")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,0.805029,0.214760
2,0.255407,0.151119
3,0.137025,0.143615
4,0.116331,0.125309


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gpt2-manim-python-finetuned/tokenizer_config.json',
 './gpt2-manim-python-finetuned/tokenizer.json')

## Testing fine-tuned model

In [17]:
import torch
model_dir = "./gpt2-manim-python-finetuned"
tokenizer = GPT2Tokenizer.from_pretrained(model_dir)
model = GPT2LMHeadModel.from_pretrained(model_dir)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [18]:
def generate_output(instruction, max_length=150, num_beams=5,
                    temperature=0.7, top_p=0.9, repetition_penalty=1.2):
  prompt = f"Instruction: {instruction}\nOutput:"
  input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

  generated_ids = model.generate(
 input_ids,
 max_length=max_length,
 num_beams=num_beams,
 temperature=temperature,
 top_p=top_p,
 repetition_penalty=repetition_penalty,
 do_sample=True,
 pad_token_id=tokenizer.eos_token_id,
 eos_token_id=tokenizer.eos_token_id,
 early_stopping=True,
 no_repeat_ngram_size=2,
  )

  generated_text = tokenizer.decode(generated_ids[0],
 skip_special_tokens=True)
  output_start = generated_text.find("Output:")
  if output_start != -1:
    output_text = generated_text[output_start + len("Output:"):].strip()
  else:
    output_text = generated_text.strip()
  return output_text

In [23]:
import csv
output_csv = "gpt2_manim_python_test_outputs.csv"
with open(output_csv, mode="w", newline="", encoding="utf-8") as csvfile:
 writer = csv.DictWriter(csvfile,
 fieldnames=["instruction", "reference_output", "generated_output"])
 writer.writeheader()

 for example in dataset['test']:
  instruction = example["instruction"]
  reference_output = example["output"]
  generated_output = generate_output(instruction)
  writer.writerow({
  "instruction": instruction,
  "reference_output": reference_output,
  "generated_output": generated_output,
  })